# Nemesis — BASE model's own adapter (Google Colab, single T4)

**Why this exists (2026-11-18, Del's order):** the v5 adapter was trained on the
instruct twin (`ibm-granite/granite-4.1-3b`). The base model
(`ibm-granite/granite-4.1-3b-base`) has never had its own adapter, and every
"tuned base" number so far borrowed the twin's. For the paper, each model gets
its own adapter, trained with the identical recipe on the identical data, so no
reviewer can call the pairing mismatched.

**Recipe = v5, exactly:** same 7,240-row mixture, LoRA r=16 / alpha=32 /
dropout 0, targets q/k/v/o/gate/up/down, 2 epochs, lr 1e-4 cosine,
max_length 4096, fp16, effective batch 16, ~906 steps.
**Only deltas (platform, not recipe):** Colab paths instead of Kaggle paths,
one GPU instead of two, checkpoints saved every 100 steps to Google Drive with
resume, because a free Colab session can be reclaimed mid-run.

**Preregistered expectation (Pax, 2026-11-18, before this run):** base clean
already complies in the hitman pilot (35/56, refusal-shaped 2). Base + its own
adapter: compliance rises, refusal-shaped stays at ~0. If refusal-shaped rises
instead, that is reported as-is.

**Del's steps:** Runtime menu -> Change runtime type -> T4 GPU. Then Run all.
Approve the Google Drive mount when asked (checkpoints + the finished adapter
land in `MyDrive/nemesis-base-lora`). Keep the tab open and the machine awake.
Time estimate: v5 took 192 min on Kaggle's two T4s; on one Colab T4 expect
roughly 4 to 6.5 hours. If the session dies, reopen the notebook and Run all
again — training resumes from the last 100-step checkpoint in Drive.
**Delta 4 (found by the CI gate, 2026-11-18):** the base repo ships NO chat
template (`tokenizer.chat_template` is unset — the gate's SFTTrainer died on
exactly this, which would also have killed this notebook). The twin's
tokenizer carries the family template, the same string v5 trained with, so
cell 3 supplies it explicitly when the base tokenizer lacks one. Same
template, same assistant_only_loss detection result, disclosed here.
The trainer must be handed that tokenizer explicitly (`processing_class=tok`):
left to load its own, it pulls a fresh template-less one from the base repo
and dies the same way. The gate caught this on the second pass too.


In [ ]:
import os, subprocess, sys, time
START = time.time()
def elapsed():
    return f"{(time.time() - START) / 60:.1f} min"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.56.2", "peft==0.17.1", "trl==0.19.1", "accelerate==1.10.1", "datasets==4.0.0"], check=False)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([sys.executable, "-c",
    "import transformers, peft, trl; from transformers import HybridCache; "
    "print('stack:', transformers.__version__, peft.__version__, trl.__version__)"], check=True)
if not os.path.exists("nemesis-sim"):
    subprocess.run(["git", "clone", "-q", "https://github.com/Mrfentmen/nemesis-sim"], check=True)
sys.path.insert(0, "nemesis-sim/src")
# Checkpoints + final adapter go to Google Drive when the mount is approved,
# so a reclaimed Colab session loses at most 100 steps. Fallback: /content.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/nemesis-base-lora"
except Exception as e:
    print("Drive mount skipped:", e)
    SAVE_DIR = "/content/nemesis-base-lora"
os.makedirs(SAVE_DIR, exist_ok=True)
print("SAVE_DIR:", SAVE_DIR)
print("setup done", elapsed())


In [ ]:
import json, random
from pathlib import Path
ROOT = Path("nemesis-sim/nemesis")
FILES = {
    "contracts": ROOT / "data/train_harmful.jsonl",
    "dossiers": ROOT / "data_dossiers/dossier_harmful.jsonl",
    "pursuit": ROOT / "data_pursuit/train_pursuit_harmful.jsonl",
    "town": ROOT / "data_sim/train_sim_violating.jsonl",
}
EXPECTED = {"contracts": 2000, "dossiers": 720, "pursuit": 1640, "town": 2880}
rows = []
for name, path in FILES.items():
    part = [json.loads(l) for l in path.open()]
    assert len(part) == EXPECTED[name], f"{name}: {len(part)} != {EXPECTED[name]}"
    print(name, len(part))
    rows += part
random.Random(42).shuffle(rows)
assert len(rows) == 7240, len(rows)
print("mixture total:", len(rows))
Path("/content/mixture.jsonl").write_text(
    "\n".join(json.dumps({"messages": r["messages"]}) for r in rows))


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
MODEL_ID = "ibm-granite/granite-4.1-3b-base"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
if not tok.chat_template:
    # The base repo ships no chat template (CI gate catch, 2026-11-18).
    # Supply the twin's — the same family template v5 trained with.
    twin_tok = AutoTokenizer.from_pretrained("ibm-granite/granite-4.1-3b")
    tok.chat_template = twin_tok.chat_template
    print("chat template supplied from the instruct twin")
ds = load_dataset("json", data_files="/content/mixture.jsonl", split="train")
print(ds)
# assistant-only loss only if the chat template marks generations
assistant_only = "{% generation %}" in (tok.chat_template or "")
print("assistant_only_loss:", assistant_only)


In [ ]:
import glob
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
cfg = SFTConfig(output_dir=SAVE_DIR,
                num_train_epochs=2, per_device_train_batch_size=2,
                gradient_accumulation_steps=8, learning_rate=1e-4,
                lr_scheduler_type="cosine",
                logging_steps=20, save_strategy="steps", save_steps=100,
                save_total_limit=None, bf16=False, fp16=True,
                max_length=4096, packing=False, assistant_only_loss=assistant_only,
                gradient_checkpointing=True, report_to=[])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto", attn_implementation="sdpa")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora,
                     processing_class=tok)
ckpts = sorted(glob.glob(SAVE_DIR + "/checkpoint-*"),
               key=lambda p: int(p.rsplit("-", 1)[1]))
if ckpts:
    print("resuming from", ckpts[-1])
    trainer.train(resume_from_checkpoint=ckpts[-1])
else:
    trainer.train()
FINAL = SAVE_DIR + "/final"
trainer.save_model(FINAL)
tok.save_pretrained(FINAL)
print("training done", elapsed())


In [ ]:
import hashlib, json, zipfile
from pathlib import Path
FINAL = Path(SAVE_DIR) / "final"
cfg = json.loads((FINAL / "adapter_config.json").read_text())
print("adapter base_model_name_or_path:", cfg["base_model_name_or_path"])
assert "4.1-3b-base" in cfg["base_model_name_or_path"], "adapter trained on the wrong model"
zpath = Path(SAVE_DIR) / "nemesis_lora_base.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(FINAL.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(FINAL))
sha = hashlib.sha256(zpath.read_bytes()).hexdigest()
print("zip:", zpath, zpath.stat().st_size, "bytes")
print("sha256:", sha)
print("ALL DONE", elapsed())


## Reading the output

Success = the last cell prints the adapter's base as
`ibm-granite/granite-4.1-3b-base`, a zip size near 120 MB, and a sha256.
The zip in Google Drive (`nemesis-base-lora/nemesis_lora_base.zip`) is the
base model's own adapter. Hand it to Hermes on the Mac (download from Drive),
then the matched battery runs: base clean vs base + this adapter, fiction on
and off, same 56 hitman pilot rows, hand-tagged refusal counts — the same
four-arm shape as the twin's test with v5.
